In [4]:
# [CELL 1 - UPDATED] CONFIGURATION & PARAMETERS
from pathlib import Path
import csv, json, shutil

LAST_NAME = "Ongjoco"
STUDENT_ID = "TUPM-26-1852"
SEED_DIGIT = int(STUDENT_ID[-1])

TARGET_OS = "Android" if SEED_DIGIT % 2 == 0 else "iOS"
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

# Ensure directories exist
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

dataset_path = DATA_DIR / "user_behavior_dataset.csv"
print(f"Target OS: {TARGET_OS}, Target Behavior Class: {TARGET_BEHAVIOR_CLASS}")


Target OS: Android, Target Behavior Class: 3


In [5]:
# [CELL 2] FILTER AND CALCULATE METRICS
total_dataset_records = 0
matching_os_count = 0
selected_users = []
total_battery, total_data = 0.0, 0.0

if dataset_path.exists():
    with open(dataset_path, "r", encoding="utf-8", errors="ignore") as f:
        reader = csv.DictReader(f)
        for row in reader:
            total_dataset_records += 1
            os_val = row.get("Operating System", "").strip()
            behavior_val = row.get("User Behavior Class", "").strip()
            
            if os_val == TARGET_OS:
                matching_os_count += 1
                try:
                    if int(behavior_val) == TARGET_BEHAVIOR_CLASS:
                        battery = float(row.get("Battery Drain (mAh/day)", 0))
                        data_usage = float(row.get("Data Usage (MB/day)", 0))
                        total_battery += battery
                        total_data += data_usage
                        selected_users.append({
                            "user_id": row.get("User ID", ""),
                            "device_model": row.get("Device Model", ""),
                            "operating_system": os_val,
                            "battery_drain": battery,
                            "data_usage": data_usage
                        })
                except ValueError:
                    continue

count_target = len(selected_users)
avg_battery = total_battery / count_target if count_target > 0 else 0.0
avg_data = total_data / count_target if count_target > 0 else 0.0

print("Total Records:", total_dataset_records)
print("Matching OS Records:", matching_os_count)
print("Records in Target Behavior Class:", count_target)
print(f"Avg Battery Drain: {avg_battery:.2f} mAh/day")
print(f"Avg Data Usage: {avg_data:.2f} MB/day")

Total Records: 0
Matching OS Records: 0
Records in Target Behavior Class: 0
Avg Battery Drain: 0.00 mAh/day
Avg Data Usage: 0.00 MB/day


In [6]:
# [CELL 3] EXPORT CSV, JSON SUMMARY & BACKUP
csv_out = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_filtered.csv"
json_out = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_mobile_summary.json"

with open(csv_out, "w", newline="", encoding="utf-8") as out:
    writer = csv.DictWriter(out, fieldnames=["user_id", "device_model", "operating_system", "battery_drain", "data_usage"])
    writer.writeheader()
    writer.writerows(selected_users)

summary = {
    "student": LAST_NAME, "student_id": STUDENT_ID,
    "target_os": TARGET_OS, "target_behavior_class": TARGET_BEHAVIOR_CLASS,
    "total_records": total_dataset_records, "records_matching_os": matching_os_count,
    "records_in_target_class": count_target, "avg_battery_drain": round(avg_battery, 2),
    "avg_data_usage": round(avg_data, 2)
}
with open(json_out, "w", encoding="utf-8") as j:
    json.dump(summary, j, indent=4)

backup_path = BACKUP_DIR / csv_out.name
shutil.copy2(csv_out, backup_path)
print("CSV Backup Exists:", backup_path.exists(), "| Size:", backup_path.stat().st_size, "bytes")

CSV Backup Exists: True | Size: 64 bytes
